# Project Sukoon — Google Colab Cloud Runner (Free T4 GPU / 12.7 GB RAM)
This notebook runs the Project Sukoon FastAPI backend and Qwen2.5-7B LLM with GPU acceleration, then exposes a public HTTPS endpoint via Cloudflare Tunnel.

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone Repository or Setup Project Files
# Replace with your GitHub repo URL if you have one, or clone the public code:
!rm -rf project-sukoon-backend
!git clone <YOUR_GITHUB_REPO_URL> project-sukoon-backend
%cd project-sukoon-backend

In [ ]:
# 3. Install Python Dependencies
!pip install -q -r requirements.txt
!pip install -q uvicorn[standard] httpx psutil

In [ ]:
# 4. Build CUDA-Accelerated llama.cpp (Runs on Nvidia T4 GPU)
%%bash
if [ ! -f "bin/llama-server" ]; then
    git clone --depth 1 https://github.com/ggml-org/llama.cpp.git /tmp/llama.cpp
    cmake -B /tmp/llama.cpp/build -S /tmp/llama.cpp -DGGML_CUDA=ON -DGGML_BUILD_TESTS=OFF -DGGML_BUILD_EXAMPLES=OFF -DGGML_BUILD_SERVER=ON
    cmake --build /tmp/llama.cpp/build --config Release -j 4 --target llama-server
    mkdir -p bin models
    cp /tmp/llama.cpp/build/bin/llama-server bin/
    chmod +x bin/llama-server
fi

# Download Qwen2.5-7B Model (~4.36 GB on Google Cloud takes ~30 seconds)
if [ ! -f "models/Qwen2.5-7B-Instruct-Q4_K_M.gguf" ]; then
    curl -L "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/qwen2.5-7b-instruct-q4_k_m.gguf" \
        -o models/Qwen2.5-7B-Instruct-Q4_K_M.gguf
fi

In [ ]:
# 5. Create .env file with your API credentials
with open('.env', 'w') as f:
    f.write('''ALPACA_API_KEY=your_key_here
ALPACA_SECRET_KEY=your_secret_here
ALPACA_PAPER_TRADING=true
FINNHUB_API_KEY=your_key_here
FMP_API_KEY=your_key_here
FRED_API_KEY=your_key_here
TELEGRAM_BOT_TOKEN=your_token_here
TELEGRAM_CHAT_ID=your_chat_id_here
LLAMA_CPP_ENDPOINT=http://127.0.0.1:8080/v1
LLAMA_CPP_MODEL_ID=Qwen2.5-7B-Instruct-Q4_K_M.gguf
LLAMA_CPP_MOCK_MODE=false
''')
print('.env written!')

In [ ]:
# 6. Launch llama-server (with 28 GPU layers offloaded to T4 VRAM) and FastAPI Backend
%%bash --bg
export PYTHONPATH=.
./bin/llama-server -m models/Qwen2.5-7B-Instruct-Q4_K_M.gguf --host 127.0.0.1 --port 8080 -c 4096 -ngl 28 > llama.log 2>&1 &
python3 -m uvicorn sukoon.api.main:app --host 127.0.0.1 --port 8000 > backend.log 2>&1 &

In [ ]:
# 7. Start Public Tunnel and Print Live HTTPS URL
!curl -sLo /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared
import subprocess, time, re

p = subprocess.Popen(['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'], stderr=subprocess.PIPE, text=True)

tunnel_url = None
for line in p.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        print('\n' + '='*60)
        print(f'🎉 SUKOON CLOUD ENDPOINT: {tunnel_url}')
        print(f'Telemetry: {tunnel_url}/api/v1/system/resources')
        print('='*60 + '\n')
        break